In [0]:
%pip install langchain
%pip install databricks-vectorsearch
dbutils.library.restartPython()

  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.6
    Not uninstalling langchain-core at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-e84b3833-7ec4-42c5-9735-532436b9507b
    Can't uninstall 'langchain-core'. No files were found to uninstall.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 12.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.3
    Not uninstalling requests at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-e84b3833-7ec4-42c5-9735-532436b9507b
    Can't uninstall 'requests'. No files were found to uninstall.
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.4
    Not uninstalling protobuf at 

In [0]:
import os
from dotenv import load_dotenv
from langchain_openai import AzureChatOpenAI
from langchain.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent
from databricks.vector_search.client import VectorSearchClient

load_dotenv("/Workspace/Users/26100305@lums.edu.pk/env")


True

In [0]:
CATALOG = "26100305-pa3"
ROLL_NUMBER = "26100305"

vsc = VectorSearchClient(
    workspace_url         = "https://dbc-facb4d62-6b45.cloud.databricks.com",
    personal_access_token = os.environ.get("DATABRICKS_TOKEN"),
    disable_notice        = True
)
endpoint_name = "26100305_vector_endpoint"
index_name = "26100305-pa3.vector_index.fixed_vector_index"
os.environ["DATABRICKS_HOST"] = "https://dbc-facb4d62-6b45.cloud.databricks.com"
os.environ["DATABRICKS_TOKEN"] = os.environ.get("DATABRICKS_TOKEN", "")

In [0]:
def retrieve_formatted_context(question: str, topk: int = 3):
    index = vsc.get_index(
        endpoint_name = endpoint_name,
        index_name    = index_name
    )
    results = index.similarity_search(
        query_text = question,
        columns    = ["ChunkText"],
        num_results = topk
    )
    docs     = results.get("result", {}).get("data_array", [])
    raw_docs = [doc[0] for doc in docs if len(doc) > 0]
    return "\n\n---\n\n".join(raw_docs)


@tool
def vector_search_tool(query: str) -> str:
    """
    Searches the Databricks Vector Store index for relevant document chunks.
    Use to retrieve context about RAG systems, SearchAgent-X, LLM efficiency,
    retrieval mechanisms, or any topic from the research paper knowledge base.
    Always use this tool first before answering any technical question.

    Args:
        query: The search query string to find relevant documents.
    Returns:
        Relevant document chunks joined by separators, or an error message.
    """
    try:
        return retrieve_formatted_context(query)
    except Exception as e:
        return f"Vector search failed: {e}"

@tool
def read_fallback_document(category: str) -> str:
    """
    Reads a comprehensive text document for a given category when vector search
    does not return relevant results. Use this tool as a fallback when the question
    is about Azure or Databricks infrastructure and vector search is insufficient.
    
    Args:
        category: The document category — must be either 'azure' or 'databricks'.
    Returns:
        Full text content of the corresponding documentation file.
    """
    file_map = {
        "databricks": f"/Volumes/{CATALOG}/default/text_documents/databricks_info.txt",
        "azure":      f"/Volumes/{CATALOG}/default/text_documents/azure_info.txt"
    }

    category  = category.lower().strip()
    file_path = file_map.get(category)

    if not file_path:
        return f"No documentation found for category: {category}"

    try:
        with open(file_path, "r") as f:
            return f.read()
    except Exception as e:
        return f"Error reading document: {e}"
    

def build_stateful_mcp_agent():
    llm = AzureChatOpenAI(
        azure_endpoint   = os.environ.get("AZURE_OPENAI_ENDPOINT"),
        azure_deployment = os.environ.get("AZURE_OPENAI_CHAT_DEPLOYMENT"),
        api_version      = os.environ.get("AZURE_OPENAI_API_VERSION"),
        api_key          = os.environ.get("AZURE_OPENAI_API_KEY"),
        temperature      = 0.0
    )

    tools  = [vector_search_tool, read_fallback_document]
    memory = MemorySaver()

    mcp_agent = create_react_agent(
        model     = llm,
        tools     = tools,
        checkpointer = memory,
        prompt    = (
            "You are a precise data engineering assistant with access to a vector "
            "knowledge base and fallback documentation.\n\n"
            "Rules:\n"
            "1. ALWAYS use vector_search_tool first to find relevant context.\n"
            "2. If vector search returns insufficient results, use read_fallback_document "
            "with category 'azure' or 'databricks'.\n"
            "3. Answer ONLY using retrieved context — never from your own knowledge.\n"
            "4. You have memory — use previous answers to answer follow-up questions."
        )
    )
    return mcp_agent

agent = build_stateful_mcp_agent()
print("Stateful MCP agent built with MemorySaver")

Stateful MCP agent built with MemorySaver


/home/spark-e84b3833-7ec4-42c5-9735-53/.ipykernel/3077/command-5392136580450899-3465088826:76: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  mcp_agent = create_react_agent(


In [0]:
# Summary: Executes queries against the agent while maintaining a consistent session thread.
def execute_mcp_queries(mcp_agent, queries: list, thread_id: str):
    config = {
        "configurable": {
            "thread_id": thread_id  # same thread_id = shared memory across queries
        }
    }

    for query in queries:
        print(f"\nUser: {query}")
        print("-" * 40)

        payload = {
            "messages": [
                {"role": "user", "content": query}
            ]
        }

        response = mcp_agent.invoke(payload, config=config)
        final_message = response["messages"][-1].content

        print(f"Agent: {final_message}")
        print("=" * 60)

In [0]:
validation_queries = [
    "What tools do you have?",
    "What is searchAgent-X?",
    "What does this agent do specifically?",
    "Use your tool to search for information about Azure.",
    "Based on those search results, summarize its main features.",
    "What is Databricks?",
    "What core features differentiate both products?"
]

print("--- TESTING STATEFUL MCP AGENT ---")
execute_mcp_queries(agent, validation_queries, "mcp_test_session_01")

--- TESTING STATEFUL MCP AGENT ---

User: What tools do you have?
----------------------------------------
Agent: I have access to the following tools:

1. vector_search_tool: Searches the Databricks Vector Store index for relevant document chunks related to RAG systems, SearchAgent-X, LLM efficiency, retrieval mechanisms, or topics from the research paper knowledge base.

2. read_fallback_document: Reads comprehensive text documents for a given category ('azure' or 'databricks') when vector search does not return relevant results. This is used for questions about Azure or Databricks infrastructure.

3. multi_tool_use.parallel: Allows running multiple tools simultaneously in parallel, specifically functions tools.

If you have any specific questions or need information, I can use these tools to find relevant context for you.

User: What is searchAgent-X?
----------------------------------------
[NOTICE] Using a Personal Authentication Token (PAT). Recommended for development only. For 

In [0]:
validation_queries = [
    "What tools do you have?",
    "What is searchAgent-X?",
    "What does this agent do specifically?",
    "Use your tool to search for information about Azure.",
    "Based on those search results, summarize its main features.",
    "What is Databricks?",
    "What core features differentiate both products?"
]

print("--- TESTING STATEFUL MCP AGENT ---")
execute_mcp_queries(agent, validation_queries, "mcp_test_session_01")

--- TESTING STATEFUL MCP AGENT ---

User: What tools do you have?
----------------------------------------
Agent: I have access to the following tools:

1. vector_search_tool: Searches the Databricks Vector Store index for relevant document chunks related to RAG systems, SearchAgent-X, LLM efficiency, retrieval mechanisms, or topics from the research paper knowledge base.

2. read_fallback_document: Reads comprehensive text documents for a given category ('azure' or 'databricks') when vector search does not return relevant results. This is used for questions about Azure or Databricks infrastructure.

3. multi_tool_use.parallel: Allows running multiple tools simultaneously in parallel, specifically functions tools.

If you have any specific questions or need information, I can use these tools to find relevant context for you.

User: What is searchAgent-X?
----------------------------------------
[NOTICE] Using a Personal Authentication Token (PAT). Recommended for development only. For 

## Project Evaluation and MCP Agent Behavior

This notebook evaluates the behavior of an MCP-enabled agent in a conversational setting. It highlights how the agent uses retrieval tools, maintains context, and responds to questions that require both knowledge lookup and reasoning.